# Avaliação de sentimentos de comentários com IMDB

Código acadêmico desenvolvido para a disciplina de **Inteligência Artificial (IA)**.

- **Professor:** Prof. Dr. Gustavo Andrade
- **Aluno:** Roberto Pereira de Freitas Neto


O Código foi totalmente escrito pelo autor, as documentações foram feitas com ajuda de IA Generativa. 

Link do repositório original escrito gradualmente pelo autor de acordo com as aulas: https://github.com/robertopfneto/imdb_commentary_evaluation/tree/main

## Instalar as dependências


In [ ]:
%pip install keras numpy tensorflow


## Vetorização


In [ ]:
import numpy as np

# cada comentario vira uma linha com 10k de posições
# a posição de uma palavra recebe: 1 - se estiver presente e 0 - se estiver ausente
# Valores atribuidos na entrada e nao nos rotulos de sentimento; Se uma palavra se repete nao aumenta o valor   

def vet_sequence(sequence, dimension):
    # cria uma linha por comentario e coluna por indice, inicialmente com zero
    results = np.zeros((len(sequence), dimension), dtype="float32")      
    for row, seq in enumerate(sequence):
        seq = [i for i in seq if i < dimension]
        results[row, seq] = 1 
    return results

## Modelo e inferência


In [ ]:
from keras import models, optimizers, layers
import re # expressao regular
from typing import List
import numpy as np



def build_model(input_data):
    model = models.Sequential([
        # recebe os valores de 10.000 posições e vai combinar seus valores com 16 neuronios com a função
        layers.Dense(16, activation="relu", input_shape=(input_data, )),  # relu remove os negativos
        layers.Dense(16, activation="relu"),  # relu remove os negativos
        # output
        layers.Dense(1, activation="sigmoid")
    ])
    
    model.compile(
        optimizer=optimizers.RMSprop(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    
    return model


# Funções para classificar texto livre

# PREPARAÇÃO
# - Limpar o texto
# - Separar palavras
# -  Procurar os codigos no dicionario de treinamento
# - gerar o vetor de presença

# REMOVE O ACENTO E PONTUAÇÃO
# o retorno p (predição) e 1 (classificação)

# Define a limpeza simplificada de uma frase em inglês e devolve uma lista de palavras.


def preprocess_text(text: str) -> List[str]:
    # Converte letras para minúsculas: Movie e movie passam a ter a mesma escrita.
    text = text.lower()
    # Substitui endereços de sites por espaços.
    text = re.sub(r"http\S+|www\S+", " ", text)
    # Substitui menções iniciadas por @ por espaços.
    text = re.sub(r"@\S+", " ", text)
    # Mantém apenas a-z e espaços; também remove pontuação, apóstrofos, dígitos e acentos.
    text = re.sub(r"[^a-z ]", " ", text)
    # Reduz espaços repetidos e remove espaços do início e do fim do texto.
    text = re.sub(r"\s+", " ", text).strip()
    # Separa a frase em palavras usando os espaços.
    tokens = text.split(" ")
    # Retorna apenas os elementos não vazios, em uma única lista.
    return [t for t in tokens if t]
# Define a conversão das palavras em códigos numéricos do vocabulário IMDB.



def text_to_imdb_indices(tokens: List[str], # Recebe o dicionário que relaciona cada palavra ao seu índice.
                         word_index: dict, # Usa por padrão o mesmo limite de vocabulário do treinamento.
                         num_words):
    
    # Cria a lista que receberá os códigos desta frase.
    indices = []
    # Percorre cada palavra produzida pela limpeza do texto.
    for token in tokens:
        # Verifica se a palavra existe no dicionário completo.
        if token in word_index:
            # Soma 3 para compatibilizar o índice com os códigos reservados do IMDB.
            idx = word_index[token] + 3
        # Trata o caso em que a palavra não existe no dicionário.
        else:
            # Usa 2 como marcador de palavra desconhecida; não representa sentimento negativo.
            idx = 2
        # Mantém somente códigos abaixo do limite; palavras conhecidas acima dele são ignoradas.
        if idx < num_words:
            # Acrescenta o código aceito à lista desta frase.
            indices.append(idx)
    # Retorna uma lista de códigos; se ficar vazia, retorna apenas o marcador 2.
    return indices or [2]
# Define a preparação completa de uma frase para a entrada da rede.


def vectorize_text(text: str, word_index: dict, num_words: int) -> np.ndarray:
    # Aplica a limpeza e separa as palavras.
    tokens = preprocess_text(text)
    # Converte as palavras em índices usando o dicionário do treinamento.
    indices = text_to_imdb_indices(tokens, word_index, num_words=num_words)
    # Retorna uma matriz com uma linha e 10.000 colunas de presença de palavras.
    return vet_sequence([indices],dimension=num_words)


# Define a classificação da frase; threshold é o limiar, por padrão 0,5.
def inference(text: str, model: dict, work_index: dict, num_words: int, threshold: float=0.5):
    # Prepara a representação numérica da frase, sem informar um gabarito à rede.
    x = vectorize_text(text, work_index, num_words=num_words)
    # Prevê sem treinar e extrai a estimativa da única linha e da única saída.
    proba = model.predict(x, verbose=0)[0, 0]
    # Usa positivo para estimativa maior ou igual a 0,5 e negativo para valor menor.
    label = "POSITIVE" if proba >= threshold else "NEGATIVE"
    # Retorna a tupla já usada no exemplo: estimativa e rótulo, para desempacotar em p e l.
    return proba, label 

## Função de treinamento


In [ ]:
# Dos 25k exemplos iniciais, 15k sao para treinamento e 10k para validacao
# epoca, batch = 512 
# Early Stopping: quantas épocas o modelo vai aguardar sem ter nenhuma melhora

def train(partial_x_train, partial_y_train, x_val, y_val, callbacks, model, epochs, batch_size):
    history = model.fit(
        # fornece os 15k comentarios do treino -> partial_x_train
        # e os routolos -> partial_y_train
        partial_x_train, partial_y_train,
        epochs=epochs,
        batch_size=batch_size,
        
        # usa validação para acompanhar o desempenho,
        validation_data = (x_val, y_val),
        callbacks=callbacks,
        # exibe o progresso e as medidas de cada epoca
        verbose=1  
    )

    return history

## Carregar e preparar o dataset


In [ ]:
from keras import callbacks
import numpy as np
import os

import tensorflow as tf # fornece os calculos que a rede neural precisa
from keras.datasets import imdb # dataset

from keras.callbacks import ModelCheckpoint
from keras.models import load_model


print("Tensorflow: ", tf.__version__)
print("Numpy: ", np.__version__)

### Carregar o dataset

NUM_WORDS = 10000 # Limitar os indices de 10.000: vocabulários frequentes
VAL_SIZE = 10000
MODEL_PATH = "text_model.keras"
EPOCHS = 20
BATCH_SIZE = 512
# rotulos: 0 - negativo, 1 - positivo. 

# carrega comentarios   
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=NUM_WORDS)

print(f"Tamanho do treino: {len(train_data)}")
print(f"Tamanho do teste: {len(test_labels)}")

# vetorização numerica dos comentarios
work_index = imdb.get_word_index() # carrega um dicionario de palavras
print(f"Palavra index -> index: {list(work_index.items())[:10]}")

# reversa Index -> palavra
reverse_word_index = {idx: word for word, idx in work_index.items()}
print(f"Indice -> palavra: {list(reverse_word_index.items())[:10]}")


# vetoriza o dataset
x_train = vet_sequence(train_data, dimension=NUM_WORDS)
x_test = vet_sequence(test_data, dimension=NUM_WORDS)

# converte os rotulos para numeros
y_train = np.asarray(train_labels, dtype="float32")
y_test = np.asarray(test_labels, dtype="float32")



## Treinar ou carregar o modelo


In [ ]:
## chamando treino

if not os.path.exists(MODEL_PATH):
    print("Pesos nao encontrados, iniciando treino...")
    model = build_model(input_data=NUM_WORDS)

    x_val, partial_x_train = x_train[:VAL_SIZE], x_train[VAL_SIZE:]
    y_val, partial_y_train = y_train[:VAL_SIZE], y_train[VAL_SIZE:]

    early_stopping = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )

    checkpont_cb = ModelCheckpoint(
        filepath=MODEL_PATH,
        save_best_only=True, # salva o melhor modelo entre as epocas
        mode="min", # min para loss e max para acuracia
        verbose=1
    )

    callbacks = [early_stopping, checkpont_cb]
    history = train(partial_x_train, partial_y_train, x_val, y_val, callbacks, model, epochs=EPOCHS, batch_size=BATCH_SIZE)

else:
    print(f"Pesos encontrados em {MODEL_PATH}")
    model = load_model(MODEL_PATH)





## Avaliar os comentários e calcular as métricas

In [ ]:
### TESTAR MODELO

# Classificar uma frase em ingles nova com a rede ja treinada
# p = estimativa(0..1) | p = predict
# l = rótulo | l = label


# atividade analise

# Comentários do teste incluídos diretamente no notebook
json_file = {
    "lista_comentarios_negativos": [
        "The Pelé movie was boring and disappointing",
        "I hated watching How to Train Your Dragon 3",
        "The Good Doctor is very bad",
        "That American movie was terrible and a waste of time",
        "I regretted taking her to see Avengers because the movie was awful"
    ],
    "lista_comentarios positivos": [
        "I should see the Pelé movie",
        "I loved watch How Train Your Dragon 3",
        "The Good Doctor is very good",
        "Maybe I will that American Movie",
        "I going to take her to see Avengers"
    ]
}

# carrega os comentarios em uma lista
lista_comentarios = [c for c in json_file.values()]


# lista_comentarios[indice_label, indice_comentario]
# 1 - positivo
# 0 - negativo

labels_positivas = lista_comentarios[1]
labels_negativas = lista_comentarios[0]

# contando verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos

TP, TN, FP, FN = 0, 0, 0, 0

lista_resultados = []

for frase in labels_positivas:
    p, l = inference(frase, model=model, work_index=work_index,  num_words=NUM_WORDS)

    if l == "NEGATIVE":
       lista_resultados.append((p, l, "FN"))
       FN+=1
        
    elif l == "POSITIVE":
       lista_resultados.append((p, l, "TP"))
       TP+=1

for frase in labels_negativas:
    p, l = inference(frase, model=model, work_index=work_index,  num_words=NUM_WORDS)

    if l == "NEGATIVE":
       lista_resultados.append((p, l, "TN"))
       TN+=1
    elif l == "POSITIVE":
       FN+=1
       lista_resultados.append((p, l, "FP"))


# exibindo resultados
print("| Comentario | Sentimento Real | Sentimento Predito | Tipo Resultado | Predict |")
for i, frase in enumerate(labels_negativas):
    print(f"{frase} | NEGATIVE | {lista_resultados[i][1]} | {lista_resultados[i][2]} | {lista_resultados[i][0]}")

for i, frase in enumerate(labels_positivas):
    print(f"{frase} | POSITIVE | {lista_resultados[i][1]} | {lista_resultados[i][2]} | {lista_resultados[i][0]}")

total = TP+TN+FP+FN
print("Total Contabilizados:" +str(total))
print(f"TP: {TP} ")
print(f"TN: {TN} ")
print(f"FP: {FP} ")
print(f"FN: {FN} ")


# calculando métricas
precisao = float(TP / (TP + FP))
recall = float(TP / (TP + FN))
f1_score = float((2*precisao*recall) / (precisao+recall))
acuracia = (TP + TN )/( TP + FN + TN + FP)
print()
print("Precisao: " +str(precisao))
print("Recall: " +str(recall))
print("F1-Score: " +str(f1_score))
print("Acurácia: " +str(acuracia))

"""
ANALISE DOS RESULTADOS

Em todas que o modelo marcou como positiva ele acertou, eh a justificativa para ter 1.0 de precisão,
entretanto, deixou passar um positivo das 5 positivas, por isso decaiu 20% na metrica de recall que ficou nas 80%,
O f1-score é uma média harmônica entre os dois casos, onde conseguimos ver que em geral, o modelo reflete uma boa capacidade
de classificar as frases, confirmado pela acurácia de 0.9

"""